# Strong and Weak References — Another 20 Advanced Problems, Fully Worked

*A step-by-step Jupyter tutorial with deliberately small executable experiments.*

This is a **new set of scenarios**, not a revision of the earlier 26-problem notebook. It follows the style of the supplied lesson: first an intuition, then a short code cell, then a written explanation, then the next experiment, and finally a test that checks the solution.

**Scope:** strong ownership and hidden strong references; weak-reference lifetime and comparison; callback order; finalizers; weak collections; reference eligibility; event subscriptions; cycles; thread coordination; and realistic cache design. All examples use the Python standard library.

**Recommended workflow:** Run cells from top to bottom. Predict the result of each stage, execute it, read the following explanation, and then change one assumption and rerun. The problems have complete solutions and assertions.

## Before we begin — reproducibility and notebook traps

The original lesson uses `ctypes.c_long.from_address(id(obj))` to look at a reference count. **Do not dereference an address after an object dies:** the address may be invalid or reused and reading it is unsafe. Here we observe lifetimes with `weakref.ref`, local scopes, assertions, and (when necessary) `gc.collect()` for reference cycles.

In CPython, non-cyclic objects often die immediately when the last strong reference disappears; the Python language does not mandate the precise timing. Tests below either retain the object deliberately, collect unreachable cycles explicitly, or test behavior within the supported CPython/Jupyter execution model. Jupyter can keep previous expression results and tracebacks alive, so we print booleans or strings, never leave a target object as the last expression in a cell. `weakref.ref(obj)()` gives a **temporary strong reference** for as long as its return value is held.

We do not assert absolute values from `sys.getrefcount`: calling it itself adds a temporary reference, and interpreter implementations differ.

In [1]:
import gc
import sys
import threading
import weakref
from collections import defaultdict
from dataclasses import dataclass
from typing import Callable

print('Python:', sys.version.split()[0])
print('Interpreter:', sys.implementation.name)
print('Ready for 20 guided problems.')

Python: 3.13.7
Interpreter: cpython
Ready for 20 guided problems.


## Problem 01 — The invisible owner in a default argument

**Your task.** Demonstrate that removing every obvious local name need not free an object if a function default argument still owns it. Then repair the lifetime.

**Mental model to use.** Default argument values are evaluated once when the function is created; the function retains them strongly.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Set up a lifetime probe

Use a function scope so Jupyter does not accidentally become an owner. The weak reference survives the scope, but the object may not.

In [2]:
def default_argument_lab():
    class Document:
        pass

    document = Document()
    witness = weakref.ref(document)

    def retrieve(saved=document):
        return saved

    del document
    return witness, retrieve

witness_1, retrieve_1 = default_argument_lab()
print('Object still alive?', witness_1() is not None)
assert witness_1() is not None

Object still alive? True


**Why is it alive?** `retrieve_1.__defaults__` contains the `Document` object. Deleting a local name removes only that binding, not the default argument held by the function. Merely calling the weak reference does not create a permanent owner unless you save its return value.

### Step 2 — Release the hidden owner

Removing the function binding removes this default-argument owner. Keep only the weak witness and check again.

In [3]:
del retrieve_1
if sys.implementation.name != 'cpython':
    gc.collect()
print('Object alive after function release?', witness_1() is not None)
assert witness_1() is None

Object alive after function release? False


### Solution: what the experiment establishes

**Diagnosis:** check function defaults and retained callables when an object outlives its apparent variables. The weak reference does not cause the retention. A production design should avoid defaulting to a large mutable object solely to cache it.

## Problem 02 — A closure that accidentally owns a session

**Your task.** Make a long-lived handler that unintentionally retains a short-lived session, then redesign it to observe the session without owning it.

**Mental model to use.** A closure holds objects captured in its cells. Capturing a `weakref.ref` is different from capturing the referent.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Create the leak-shaped closure

Here the handler reads `session.name`, so it captures `session` strongly.

In [4]:
class SessionForClosure:
    def __init__(self, name):
        self.name = name


def strong_handler_factory():
    session = SessionForClosure('session-A')
    observer = weakref.ref(session)

    def handler():
        return session.name

    return observer, handler

session_watch, strong_handler = strong_handler_factory()
print('Retained by handler?', session_watch() is not None)
assert strong_handler() == 'session-A'

Retained by handler? True


Inspecting closure cells can reveal retention, but **do not store the cell contents for debugging**: doing so can create another strong owner. Here the handler itself is the reason the session remains reachable.

### Step 2 — Remove the unintended owner

Discard the handler and verify its captured session becomes unreachable.

In [5]:
del strong_handler
if sys.implementation.name != 'cpython':
    gc.collect()
assert session_watch() is None
print('After releasing handler: session collected.')

After releasing handler: session collected.


### Step 3 — Build a weak-capturing handler

The replacement closure captures only a weak reference. It safely dereferences once and handles absence explicitly.

In [6]:
def weak_handler_factory():
    session = SessionForClosure('session-B')
    observer = weakref.ref(session)

    def handler():
        current = observer()
        if current is None:
            return '<session expired>'
        return current.name

    assert handler() == 'session-B'
    return observer, handler

session_watch_2, weak_handler = weak_handler_factory()
assert session_watch_2() is None
assert weak_handler() == '<session expired>'
print(weak_handler())

<session expired>


### Solution: what the experiment establishes

A callback that should *observe* a resource must not capture that resource strongly. Call the weak ref once, keep the returned object in a local while using it, then let the local go out of scope.

## Problem 03 — A suspended generator keeps a payload alive

**Your task.** Show that generator frames retain local variables across `yield`, even when the caller drops its payload variable. Release that state deliberately.

**Mental model to use.** A paused generator owns its execution frame, including live locals.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Pause a generator after creating the payload

Yield only a plain string, never the payload itself; otherwise the notebook output could retain it.

In [7]:
def make_suspended_generator():
    class Payload:
        pass

    payload = Payload()
    witness = weakref.ref(payload)

    def stream(held=payload):
        local_payload = held
        yield 'started'
        return local_payload is not None

    generator = stream()
    del payload  # The generator frame now owns its bound default argument.
    return witness, generator

watch_generator, generator_3 = make_suspended_generator()
assert next(generator_3) == 'started'
print('Alive while paused?', watch_generator() is not None)
assert watch_generator() is not None

Alive while paused? True


A suspended frame is state, not just executable code. Here the generator frame retains the object through its bound default argument. Removing the external payload variable was not enough; the generator still has state that owns it.

### Step 2 — Close and release the generator

Calling `close()` stops a paused generator and releases its frame, but other strong owners must also be absent. This factory leaves no external closure reference except the generator itself.

In [8]:
generator_3.close()
del generator_3
if sys.implementation.name != 'cpython':
    gc.collect()
assert watch_generator() is None
print('Payload released after generator teardown.')

Payload released after generator teardown.


### Solution: what the experiment establishes

When auditing memory, inspect suspended generators, coroutines, and long-lived iterators. `close()` is a useful cleanup operation for generators, but release any other strong aliases too.

## Problem 04 — A saved exception traceback retains a local object

**Your task.** Reproduce retention by keeping an exception with its traceback, then break the chain without relying on a REPL’s exception history.

**Mental model to use.** An exception refers to its traceback; traceback frames can retain their local variables.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Raise while holding a payload

Catch and return the exception from an isolated function. Its traceback preserves the raising frame and its local `payload`.

In [9]:
class TracePayload:
    pass


def traceback_factory():
    payload = TracePayload()
    observer = weakref.ref(payload)
    try:
        raise RuntimeError('demonstration failure')
    except RuntimeError as exc:
        saved = exc
    return observer, saved

trace_watch, saved_exc = traceback_factory()
assert trace_watch() is not None
print('Retained by traceback?', trace_watch() is not None)

Retained by traceback? True


The exception can be valuable for diagnostics yet costly if stored indefinitely. You should not use `except ... as exc` outside this scoped experiment to measure lifetime, because traceback references and notebook history may distort results.

### Step 2 — Cut the traceback edge, then discard the exception

Setting `__traceback__` to `None` on this exception removes its traceback chain; this lab has no exception cause or context that holds the payload.

In [10]:
saved_exc.__traceback__ = None
assert trace_watch() is None
del saved_exc
print('Traceback cleared; payload released.')

Traceback cleared; payload released.


### Solution: what the experiment establishes

Storing exceptions, traceback objects, or frames in application-wide error histories may keep large local data alive. Preserve a formatted diagnostic string if retaining executable frame state is unnecessary.

## Problem 05 — Weak-reference equality changes when targets die

**Your task.** Investigate weak-reference equality and hashing before and after collection without relying on object addresses.

**Mental model to use.** Live weak references compare based on their referents; dead weak references compare equal only to themselves. Hashing uses the referent hash and can be unavailable if first attempted after death.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Create two distinct reference objects

Provide different callbacks so two different `weakref.ref` objects are created for the same referent. Keep both references strongly.

In [11]:
class HashTarget:
    pass

hash_events = []
target_5 = HashTarget()
ref_5a = weakref.ref(target_5, lambda ref: hash_events.append('A'))
ref_5b = weakref.ref(target_5, lambda ref: hash_events.append('B'))
assert ref_5a is not ref_5b
assert ref_5a == ref_5b
stored_hash = hash(ref_5a)
assert stored_hash == hash(ref_5b)
print('Live references compare equal and share a hash.')

Live references compare equal and share a hash.


### Step 2 — Collect the target and repeat the comparisons

The references remain independently allocated objects. Once dead, they no longer compare equal to *each other*, even though they formerly had the same referent.

In [12]:
del target_5
assert ref_5a() is None and ref_5b() is None
assert ref_5a != ref_5b
assert ref_5a == ref_5a
assert hash(ref_5a) == stored_hash
print('Dead refs: self-equality yes, distinct-ref equality no.')

Dead refs: self-equality yes, distinct-ref equality no.


### Step 3 — Attempt the first hash only after death

Build another weak reference but intentionally do not hash it until its object is gone.

In [13]:
late_target = HashTarget()
late_ref = weakref.ref(late_target)
del late_target
try:
    hash(late_ref)
except TypeError as error:
    print(type(error).__name__ + ': hashing first attempted too late')
else:
    raise AssertionError('Expected TypeError for first hash after collection')

TypeError: hashing first attempted too late


### Solution: what the experiment establishes

Weak references are not interchangeable with stable identity tokens or integer IDs. If you must hash a weak reference, do so while the referent is live; avoid building an identity registry that assumes weak-ref equality stays unchanged across death.

## Problem 06 — What order do weakref callbacks run in?

**Your task.** Register several callbacks on one referent and establish which runs first when it dies, without keeping it alive through callback closures.

**Mental model to use.** The `weakref` documentation specifies callback invocation from most recently registered to oldest, for weak references that remain alive.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Register three callbacks with separate labels

Capture only each *label*, not the observed object. Keep the reference objects alive in a list.

In [14]:
callback_order = []

class Signal:
    pass

signal_6 = Signal()
callbacks_6 = []
for label in ('first', 'second', 'third'):
    callbacks_6.append(
        weakref.ref(signal_6, lambda dead_ref, label=label: callback_order.append(label))
    )
assert len(callbacks_6) == 3
print('Registered: first, second, third')

Registered: first, second, third


### Step 2 — Release the subject

The callbacks receive weak-reference objects, not a live referent. Their return values are ignored.

In [15]:
del signal_6
assert callback_order == ['third', 'second', 'first']
assert all(ref() is None for ref in callbacks_6)
print('Fired:', ' → '.join(callback_order))

Fired: third → second → first


### Solution: what the experiment establishes

Rely on this ordering only when it is truly useful; independent cleanup callbacks should generally not depend on one another. A callback must be short, defensive, and avoid capturing the referent.

## Problem 07 — A callback must not break cleanup when logging fails

**Your task.** Design a weakref callback that records cleanup but handles its own errors instead of raising into the interpreter’s unraisable-exception machinery.

**Mental model to use.** Exceptions escaping weakref callbacks cannot be propagated to the code that caused collection; treat callback boundaries like destructor boundaries.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Write a callback with an explicitly handled failure

Simulate a logger failure. The callback catches it and records a plain error string; no exception leaves the callback.

In [16]:
callback_log = []


def safe_callback(dead_ref):
    try:
        callback_log.append('cleanup began')
        raise OSError('simulated logging outage')
    except Exception as exc:
        callback_log.append('handled: ' + type(exc).__name__)

class ShortJob:
    pass

job_7 = ShortJob()
job_ref_7 = weakref.ref(job_7, safe_callback)
print('Before deletion:', callback_log)

Before deletion: []


### Step 2 — Verify exactly one cleanup notification

The event log is plain data and does not own the `ShortJob`.

In [17]:
del job_7
assert job_ref_7() is None
assert callback_log == ['cleanup began', 'handled: OSError']
print(' | '.join(callback_log))

cleanup began | handled: OSError


### Solution: what the experiment establishes

Avoid exceptions, blocking I/O, object resurrection tricks, and complex dependency ordering inside callbacks. Prefer explicit lifecycle methods for essential cleanup; weakref callbacks should be best-effort fallbacks.

## Problem 08 — Cancel a registered finalizer with `detach`

**Your task.** Use `weakref.finalize` to model an optional fallback action, inspect its state, and cancel it before collection.

**Mental model to use.** `finalize` retains the cleanup function and its arguments; `detach()` removes and returns that registration without calling it.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Register a finalizer without capturing its subject

The cleanup function receives only a plain identifier and output list, neither of which points back to the owner.

In [18]:
finalizer_events_8 = []

class TemporaryReport:
    pass

report_8 = TemporaryReport()
finalizer_8 = weakref.finalize(
    report_8, finalizer_events_8.append, 'report-closed'
)
assert finalizer_8.alive
print('Finalizer active?', finalizer_8.alive)

Finalizer active? True


### Step 2 — Inspect and detach

`peek()` returns registration details while active. Do not save its return value: it contains the object, and saving it could change its lifetime.

In [19]:
assert finalizer_8.peek() is not None
registration_8 = finalizer_8.detach()
assert registration_8 is not None
# Registration tuple includes the referent; discard it before testing expiration.
del registration_8
assert not finalizer_8.alive
print('Detached; action will not run automatically.')

Detached; action will not run automatically.


### Step 3 — Drop the owner and verify cancellation

The original `finalize` object remains, but it is now inactive.

In [20]:
del report_8
assert finalizer_events_8 == []
assert finalizer_8.peek() is None
print('No cleanup callback ran after detach.')

No cleanup callback ran after detach.


### Solution: what the experiment establishes

`detach()` is useful when ownership of cleanup moves elsewhere. By contrast, calling a live finalizer directly executes its action once. `peek()` and `detach()` expose the referent temporarily: discard those tuples before lifetime tests.

## Problem 09 — Explicit resource release plus a finalizer fallback

**Your task.** Implement a resource lease whose explicit `close()` is idempotent and whose fallback does not accidentally keep the lease alive.

**Mental model to use.** Put cleanup state outside the owner; register a function that closes that state, not a bound method of the owner.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Build an independent resource state

The state is intentionally simple: a boolean and a list of string events. The owner owns it; the finalizer also holds it, but it never refers back to the owner.

In [21]:
class ResourceState:
    def __init__(self, name, events):
        self.name = name
        self.events = events
        self.closed = False

    def close(self):
        if not self.closed:
            self.closed = True
            self.events.append('closed:' + self.name)


class ResourceLease:
    def __init__(self, name, events):
        self._state = ResourceState(name, events)
        self._finalizer = weakref.finalize(self, ResourceState.close, self._state)

    def close(self):
        self._finalizer()  # invokes once; marks the finalizer dead

    @property
    def closed(self):
        return self._state.closed

### Step 2 — Test explicit close and repeat calls

Idempotence matters: application code and fallback mechanisms may attempt to close at different times.

In [22]:
events_9 = []
lease_9 = ResourceLease('explicit', events_9)
lease_9.close()
lease_9.close()
assert lease_9.closed
assert events_9 == ['closed:explicit']
print(events_9)

['closed:explicit']


### Step 3 — Test fallback without storing the lease

The witness observes the lease. Its finalizer runs when the lease is collected, and the independent state records cleanup.

In [23]:
fallback_events_9 = []
fallback_lease_9 = ResourceLease('fallback', fallback_events_9)
fallback_watch_9 = weakref.ref(fallback_lease_9)
del fallback_lease_9
assert fallback_watch_9() is None
assert fallback_events_9 == ['closed:fallback']
print(fallback_events_9)

['closed:fallback']


### Solution: what the experiment establishes

Prefer explicit `close()` (or a context manager) for timely external resource release. A finalizer is a safety net, not a guaranteed timely substitute. Registering `self.close` as the cleanup callback would retain `self` and defeat the lifetime design.

## Problem 10 — Replacing a weak cache entry while the old value lives

**Your task.** Verify that expiration of a formerly cached value does not erase a newer value installed under the same key.

**Mental model to use.** `WeakValueDictionary` stores weak references to values. Its cleanup must recognize which weak reference currently owns a key.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Create a weak-value cache and insert v1

Retain the values in explicit local variables while testing. Plain strings cannot generally serve as weakly referenced values.

In [24]:
class CacheValue:
    def __init__(self, version):
        self.version = version

cache_10 = weakref.WeakValueDictionary()
old_10 = CacheValue('v1')
cache_10['alpha'] = old_10
assert cache_10['alpha'] is old_10
print('Installed version:', cache_10['alpha'].version)

Installed version: v1


### Step 2 — Replace v1 with v2

Both objects are still alive, but only v2 is the cache entry.

In [25]:
new_10 = CacheValue('v2')
cache_10['alpha'] = new_10
assert cache_10['alpha'] is new_10
old_watch_10 = weakref.ref(old_10)
print('Replacement installed:', cache_10['alpha'].version)

Replacement installed: v2


### Step 3 — Destroy v1, then inspect v2

The old-value cleanup must not remove the replacement.

In [26]:
del old_10
assert old_watch_10() is None
assert cache_10['alpha'] is new_10
print('Old version died; replacement survived.')
del new_10
assert 'alpha' not in cache_10
print('Replacement eventually expired too.')

Old version died; replacement survived.
Replacement eventually expired too.


### Solution: what the experiment establishes

Weak-value eviction is about **object ownership**, not a cache time-to-live. Keep a strong owner for values that must stay resident; verify replacement and expiration transitions independently.

## Problem 11 — Why a weak cache may recompute immediately

**Your task.** Write a cache lookup function that can successfully return a newly built object while the cache entry disappears soon after the caller discards it.

**Mental model to use.** A `WeakValueDictionary` does not own the value it stores; the returned result is a temporary strong owner.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Write an explicitly non-owning memoizer

Count builds in a plain list. The factory creates a custom weak-referenceable class instance.

In [27]:
class Computation:
    def __init__(self, key):
        self.key = key

build_log_11 = []
weak_cache_11 = weakref.WeakValueDictionary()


def lookup_11(key):
    item = weak_cache_11.get(key)
    if item is None:
        build_log_11.append(key)
        item = Computation(key)
        weak_cache_11[key] = item
    return item

### Step 2 — Demonstrate reuse while a caller owns the object

A live caller variable makes the cache effective for repeated queries.

In [28]:
owned_11 = lookup_11('model')
again_11 = lookup_11('model')
assert owned_11 is again_11
assert build_log_11 == ['model']
print('Builds while held:', len(build_log_11))

Builds while held: 1


### Step 3 — Remove ownership and try again

Observe the weak reference without saving the object returned by calling it.

In [29]:
witness_11 = weakref.ref(owned_11)
del owned_11, again_11
assert witness_11() is None
assert 'model' not in weak_cache_11
third_11 = lookup_11('model')
assert build_log_11 == ['model', 'model']
print('Builds after release:', len(build_log_11))

Builds after release: 2


### Solution: what the experiment establishes

Weak-value caching helps reuse objects already owned elsewhere; it does not guarantee reuse across independent calls. If residency is required, add deliberate strong retention, e.g. a bounded LRU, alongside weak observations.

## Problem 12 — Mutating a WeakSet during iteration

**Your task.** Explain the difference between iterating a live weak collection and iterating a stable snapshot while callbacks unsubscribe.

**Mental model to use.** Weak collections can change as targets die or as your own loop removes entries. Materialize a strong snapshot for a stable iteration.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Create live listeners

These objects are hashable by identity and support weak references.

In [30]:
class LiveListener:
    def __init__(self, name):
        self.name = name

listeners_12 = [LiveListener('A'), LiveListener('B'), LiveListener('C')]
set_12 = weakref.WeakSet(listeners_12)
assert len(set_12) == 3
print('Live listeners:', len(set_12))

Live listeners: 3


### Step 2 — Make the unsafe pattern concrete

The Python iterator may reject explicit set mutation during traversal. We deliberately avoid asserting which entry it visits first.

In [31]:
try:
    for listener in set_12:
        set_12.discard(listener)
except RuntimeError as error:
    print('Live iteration failed:', type(error).__name__)
else:
    print('This run did not raise; still avoid mutating a live collection.')

Live iteration failed: RuntimeError


### Step 3 — Use a snapshot and rebuild the fixture

`list(set_12)` temporarily owns every listener it contains; this is intentional for the duration of the notification pass.

In [32]:
set_12 = weakref.WeakSet(listeners_12)
visited_12 = []
for listener in list(set_12):
    visited_12.append(listener.name)
    set_12.discard(listener)
assert sorted(visited_12) == ['A', 'B', 'C']
assert len(set_12) == 0
print('Snapshot processed:', sorted(visited_12))

Snapshot processed: ['A', 'B', 'C']


### Solution: what the experiment establishes

Snapshot iteration trades temporary strong retention for predictable traversal. Do not rely on a WeakSet’s iteration order; assert sets or sorted names instead. Concurrent mutation also needs synchronization.

## Problem 13 — An event bus supporting functions and bound methods

**Your task.** Design a minimal subscription mechanism that weakly tracks instances for bound methods while retaining ordinary functions under an explicit policy.

**Mental model to use.** A bound method is an ephemeral wrapper; `WeakMethod` tracks its instance and function. Ordinary functions are often best kept strongly by a bus that promises subscription.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Create a subscription wrapper

Separate the two ownership policies. The bus keeps `WeakMethod` objects but directly stores standalone functions.

In [33]:
class TutorialEventBus:
    def __init__(self):
        self._methods = []
        self._functions = []

    def subscribe(self, callback):
        if getattr(callback, '__self__', None) is not None:
            self._methods.append(weakref.WeakMethod(callback))
        else:
            self._functions.append(callback)

    def publish(self, message):
        self._methods = [r for r in self._methods if r() is not None]
        # Use a snapshot: callbacks can add/remove subscribers in real systems.
        for fn in list(self._functions):
            fn(message)
        for ref in list(self._methods):
            fn = ref()     # one dereference; keep method alive during invocation
            if fn is not None:
                fn(message)

    @property
    def subscription_count(self):
        self._methods = [r for r in self._methods if r() is not None]
        return len(self._functions) + len(self._methods)

**Design choice:** a strongly stored standalone function stays subscribed until explicitly removed; a bound-method subscription ends when its instance dies. A complete production bus would return unsubscribe tokens and define thread safety, exceptions, ordering, and duplicate policies. This lab isolates reference ownership.

### Step 2 — Install subscribers and verify dispatch

Keep a subscriber alive explicitly and verify both subscriber categories receive the event.

In [34]:
events_13 = []

class Receiver:
    def __init__(self, label):
        self.label = label

    def on_event(self, message):
        events_13.append(self.label + ':' + message)


def global_receiver(message):
    events_13.append('global:' + message)

bus_13 = TutorialEventBus()
bus_13.subscribe(global_receiver)
receiver_13 = Receiver('instance')
receiver_watch_13 = weakref.ref(receiver_13)
bus_13.subscribe(receiver_13.on_event)
bus_13.publish('hello')
assert set(events_13) == {'global:hello', 'instance:hello'}
print(sorted(events_13))

['global:hello', 'instance:hello']


### Step 3 — Remove the subscriber instance

The bus keeps a weak method, not `receiver_13` itself. Publish again and check cleanup.

In [35]:
del receiver_13
assert receiver_watch_13() is None
bus_13.publish('again')
assert bus_13.subscription_count == 1
assert events_13.count('global:again') == 1
assert 'instance:again' not in events_13
print('Remaining subscriptions:', bus_13.subscription_count)

Remaining subscriptions: 1


### Solution: what the experiment establishes

Weak subscriptions are appropriate only when the system should *not own* subscribers. A bus that silently drops every lambda/function may violate application expectations; choose and document an explicit ownership policy.

## Problem 14 — Built-in containers and subclass weakref support

**Your task.** Test which built-in instances support weak references, and contrast ordinary containers with subclass instances.

**Mental model to use.** Weak-reference eligibility depends on an object’s type and layout, not on whether it is mutable or hashable.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Probe without treating exceptions as notebook failures

The function returns a boolean and never exposes or stores the referent.

In [36]:
def supports_weakref(value):
    try:
        weakref.ref(value)
    except TypeError:
        return False
    return True

plain_list_14 = []
plain_dict_14 = {}
plain_int_14 = 42
print({
    'list': supports_weakref(plain_list_14),
    'dict': supports_weakref(plain_dict_14),
    'int': supports_weakref(plain_int_14),
})
assert not any(map(supports_weakref, (plain_list_14, plain_dict_14, plain_int_14)))

{'list': False, 'dict': False, 'int': False}


### Step 2 — Subclass list and dict

Instances of ordinary Python subclasses of these two container types can support weak references. Test explicitly; do not generalize to `int` or `tuple` subclasses.

In [37]:
class WeakList(list):
    pass

class WeakDict(dict):
    pass

custom_list_14 = WeakList([1, 2])
custom_dict_14 = WeakDict({'x': 1})
assert supports_weakref(custom_list_14)
assert supports_weakref(custom_dict_14)
print('list subclass:', supports_weakref(custom_list_14))
print('dict subclass:', supports_weakref(custom_dict_14))

list subclass: True
dict subclass: True


### Step 3 — Separate weak-referenceability from hashability

A list subclass can be weakly referenced yet remain unhashable: the key requirement for WeakKeyDictionary is stronger.

In [38]:
assert weakref.ref(custom_list_14)() is custom_list_14
try:
    hash(custom_list_14)
except TypeError:
    print('Weak-referenceable does NOT imply hashable.')
else:
    raise AssertionError('A normal list subclass should remain unhashable')

Weak-referenceable does NOT imply hashable.


### Solution: what the experiment establishes

For `WeakKeyDictionary`, an instance must be both weak-referenceable **and** hashable. `WeakValueDictionary` requires weak-referenceable values, but its external keys follow ordinary dict hashability rules.

## Problem 15 — Slot inheritance and accidental weakref eligibility

**Your task.** Compare a fully slotted class, an explicit `__weakref__` slot, and a subclass that reintroduces an instance dictionary.

**Mental model to use.** An ordinary subclass without its own slots often gains `__dict__` and `__weakref__`; a slotted base alone does not force all descendants to lack them.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Create three related layouts

A direct `object` subclass with nonempty `__slots__` and no `__weakref__` typically rejects weak references.

In [39]:
class SlottedBase:
    __slots__ = ('value',)

    def __init__(self, value):
        self.value = value


class ExplicitWeakSlot:
    __slots__ = ('value', '__weakref__')

    def __init__(self, value):
        self.value = value


class DynamicChild(SlottedBase):
    pass

base_15 = SlottedBase(1)
opted_in_15 = ExplicitWeakSlot(2)
child_15 = DynamicChild(3)
assert not supports_weakref(base_15)
assert supports_weakref(opted_in_15)
assert supports_weakref(child_15)
print('base / opt-in / child:', False, True, True)

base / opt-in / child: False True True


### Step 2 — Preserve the layout across subclasses

A subclass with `__slots__ = ()` does not automatically reintroduce a dictionary or weak-reference support.

In [40]:
class StillSlottedChild(SlottedBase):
    __slots__ = ()

still_slotted_15 = StillSlottedChild(4)
assert not supports_weakref(still_slotted_15)
print('Slotted child weak-referenceable?', supports_weakref(still_slotted_15))

Slotted child weak-referenceable? False


### Solution: what the experiment establishes

Check the **actual concrete instance type** before designing a weak-key storage layer. Declaring `__weakref__` is useful for a slotted class only when the base hierarchy has not already provided weak-reference support.

## Problem 16 — Inspecting weak references without manufacturing owners

**Your task.** Use `weakref.getweakrefcount` and `weakref.getweakrefs` to inspect an instance while recognizing that the returned list itself owns reference objects, not the referent.

**Mental model to use.** Weak-reference introspection reports live weakref objects and proxy objects; exact counts can differ when plain weakref instances are reused internally.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Create references that must be kept alive

Use callbacks to force separate weak reference objects; callback labels capture no observed object.

In [41]:
class InspectTarget:
    pass

inspect_16 = InspectTarget()
refs_16 = [
    weakref.ref(inspect_16, lambda ref, label=i: None)
    for i in range(3)
]
count_16 = weakref.getweakrefcount(inspect_16)
assert count_16 >= 3
print('Tracked weak references:', count_16)

Tracked weak references: 3


### Step 2 — Inspect the reference objects

The list from `getweakrefs` is not a list of owned targets; avoid dereferencing entries and storing referents just for debugging.

In [42]:
known_refs_16 = weakref.getweakrefs(inspect_16)
assert all(isinstance(ref, weakref.ReferenceType) for ref in known_refs_16)
assert set(refs_16).issubset(set(known_refs_16))
print('Retrieved reference objects:', len(known_refs_16))
del known_refs_16

Retrieved reference objects: 3


### Step 3 — Dispose of the target without disposing of its observers

Weak-reference objects remain in `refs_16` but their targets no longer exist.

In [43]:
del inspect_16
assert all(ref() is None for ref in refs_16)
print('All saved reference objects are now dead.')

All saved reference objects are now dead.


### Solution: what the experiment establishes

Use introspection as a diagnostic, not a strong-reference counting oracle. Weakref counts can include machinery installed by other containers and vary across implementation details.

## Problem 17 — A subscriber token that mistakenly captures its owner

**Your task.** Make an unregister token that is safe to keep after the subscriber dies. Contrast it with the common closure-capture mistake.

**Mental model to use.** An “unsubscribe” closure that references the subscriber strongly defeats weak registration; use a weak reference in the token.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Reproduce a retained subscriber

The subscription registry keeps only weak references, but the returned token closes over the subscriber.

In [44]:
class MiniSubscriber:
    pass

bad_registry_17 = weakref.WeakSet()


def bad_register_17(subscriber):
    bad_registry_17.add(subscriber)

    def unsubscribe():
        bad_registry_17.discard(subscriber)  # retains subscriber strongly

    return unsubscribe

subscriber_17 = MiniSubscriber()
watch_17 = weakref.ref(subscriber_17)
bad_token_17 = bad_register_17(subscriber_17)
del subscriber_17
assert watch_17() is not None
print('Bad token retains subscriber:', watch_17() is not None)

Bad token retains subscriber: True


### Step 2 — Repair the ownership boundary

Use one weak reference as the token’s captured object. Dereference once when the token is called.

In [45]:
del bad_token_17
assert watch_17() is None

good_registry_17 = weakref.WeakSet()


def good_register_17(subscriber):
    good_registry_17.add(subscriber)
    observer = weakref.ref(subscriber)

    def unsubscribe():
        current = observer()
        if current is not None:
            good_registry_17.discard(current)

    return unsubscribe

subscriber_17b = MiniSubscriber()
watch_17b = weakref.ref(subscriber_17b)
good_token_17 = good_register_17(subscriber_17b)
del subscriber_17b
assert watch_17b() is None
good_token_17()   # safely no-ops after collection
print('Good token does not retain subscriber.')

Good token does not retain subscriber.


### Solution: what the experiment establishes

Review **all** registered functions and returned tokens for captured owners. A weak collection alone cannot guarantee non-ownership if one of its values, callbacks, or nearby closures points back to the object.

## Problem 18 — A cyclic graph: callback versus reachability

**Your task.** Construct a two-node ownership cycle and use a weakref callback to detect eventual collection without storing either node in the callback.

**Mental model to use.** Reference counting cannot alone reclaim an unreachable strong cycle. The cyclic garbage collector can discover and reclaim it.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Build a cycle inside a function

The function returns only a weak observer. The callback retains an events list but no graph node.

In [46]:
cycle_events_18 = []

class GraphNode:
    def __init__(self, label):
        self.label = label
        self.next = None


def build_cycle_18(events):
    left = GraphNode('left')
    right = GraphNode('right')
    left.next = right
    right.next = left
    watcher = weakref.ref(left, lambda dead_ref: events.append('left-reclaimed'))
    return watcher

watch_cycle_18 = build_cycle_18(cycle_events_18)
print('Observer created; two nodes point to each other.')

Observer created; two nodes point to each other.


### Step 2 — Ask the collector to discover unreachable cycles

`gc.collect()` can also collect unrelated unreachable objects, so do not assert the global count. Assert only our observer and callback.

In [47]:
gc.collect()
assert watch_cycle_18() is None
assert cycle_events_18 == ['left-reclaimed']
print('Callback:', cycle_events_18[0])

Callback: left-reclaimed


### Step 3 — Collect again and verify exactly-once behavior

A weak-reference callback associated with a given weak reference is not called twice for the same object death.

In [48]:
gc.collect()
assert cycle_events_18 == ['left-reclaimed']
print('No duplicate callback after another collection.')

No duplicate callback after another collection.


### Solution: what the experiment establishes

Cycles change *when* an unreachable target disappears, not the meaning of a weak reference. `gc.collect()` is a test aid here, not a general substitute for proper resource-lifecycle design.

## Problem 19 — Coordinate a weak-value cache across two threads

**Your task.** Implement a thread-safe `get_or_create` so two workers requesting the same key share one live object, without making the cache its permanent owner.

**Mental model to use.** Use a lock around the check/build/insert operation. Callers must retain returned objects while reuse matters.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Create the cache and model

A `Lock` makes the lookup atomic relative to other calls to this cache. We deliberately use a single coarse lock to keep the correctness proof simple.

In [49]:
class SharedResult:
    def __init__(self, key, serial):
        self.key = key
        self.serial = serial


class ThreadSafeWeakCache:
    def __init__(self):
        self._values = weakref.WeakValueDictionary()
        self._lock = threading.Lock()
        self.builds = 0

    def get_or_create(self, key):
        with self._lock:
            item = self._values.get(key)
            if item is None:
                self.builds += 1
                item = SharedResult(key, self.builds)
                self._values[key] = item
            return item

### Step 2 — Keep both workers’ results alive for inspection

The list is intentional strong ownership. Starting threads is not a guarantee of overlap, but the lock guarantees correct behavior for either schedule.

In [50]:
cache_19 = ThreadSafeWeakCache()
results_19 = []
results_lock_19 = threading.Lock()
start_19 = threading.Barrier(3)


def worker_19():
    start_19.wait()
    result = cache_19.get_or_create('same-key')
    with results_lock_19:
        results_19.append(result)

threads_19 = [threading.Thread(target=worker_19) for _ in range(2)]
for thread in threads_19:
    thread.start()
start_19.wait()
for thread in threads_19:
    thread.join()
assert len(results_19) == 2
assert results_19[0] is results_19[1]
assert cache_19.builds == 1
print('Concurrent callers shared one build.')

Concurrent callers shared one build.


### Step 3 — Remove caller ownership

Once callers release their results, this weak cache may evict its entry. Build again and verify the behavior is intentional.

In [51]:
watch_19 = weakref.ref(results_19[0])
del results_19
assert watch_19() is None
new_result_19 = cache_19.get_or_create('same-key')
assert cache_19.builds == 2
print('Builds after callers released result:', cache_19.builds)

Builds after callers released result: 2


### Solution: what the experiment establishes

Locking protects the cache operation, not the lifetime of a returned result after the caller releases it. For expensive factories, a coarse lock also serializes unrelated keys; production code may use per-key synchronization and explicit exception policies.

## Problem 20 — Capstone: a non-owning registry with safe subscription tokens

**Your task.** Combine weak registrations, snapshot iteration, callback failure isolation, automatic expiry, and idempotent explicit unsubscribe. Deliver messages to live clients without owning them.

**Mental model to use.** Registry entries are `weakref.ref` values keyed by monotonic integers; tokens capture only a weak reference to the registry plus an integer token. The registry itself must be weak-referenceable.

Work through the stages before checking the final assertion. Each stage is deliberately short, like the supplied lesson.

### Step 1 — Implement registry and token

Unlike a `WeakSet`, integer tokens distinguish subscriptions even when objects define unusual equality. The registry stores only weak references to clients and does not hold a client through the cleanup callback.

In [52]:
class WeakClientRegistry:
    def __init__(self):
        self._entries = {}
        self._next_id = 0
        self.errors = []

    def subscribe(self, client):
        self._next_id += 1
        token_id = self._next_id
        owner_ref = weakref.ref(self)

        def on_client_death(dead_ref):
            registry = owner_ref()
            if registry is not None:
                registry._entries.pop(token_id, None)

        self._entries[token_id] = weakref.ref(client, on_client_death)

        def unsubscribe():
            registry = owner_ref()
            if registry is not None:
                registry._entries.pop(token_id, None)

        return unsubscribe

    def publish(self, message):
        # Freeze the references, not their referents. Each live client is
        # acquired once and held strongly for its own invocation.
        for token_id, observer in list(self._entries.items()):
            client = observer()
            if client is None:
                self._entries.pop(token_id, None)
                continue
            try:
                client.receive(message)
            except Exception as error:
                self.errors.append((token_id, type(error).__name__))

    @property
    def count(self):
        return len(self._entries)

**Why this ownership design works:** the registry owns observer objects, not client objects. Its death callback closes over `owner_ref` (a weak reference to the registry) and an integer, not the registry or client themselves. Tokens have the same non-owning property. We snapshot dictionary items so client callbacks can unsubscribe safely during publication. Error policy is to record exception types and continue.

### Step 2 — Create a normal client and a failing client

The registry should call both, catch one failure, and never retain either client as a consequence of dispatch.

In [53]:
delivery_log_20 = []

class RegisteredClient:
    def __init__(self, name, fail=False):
        self.name = name
        self.fail = fail

    def receive(self, message):
        if self.fail:
            raise ValueError('simulated client failure')
        delivery_log_20.append(self.name + ':' + message)

registry_20 = WeakClientRegistry()
alice_20 = RegisteredClient('Alice')
bob_20 = RegisteredClient('Bob', fail=True)
watch_alice_20 = weakref.ref(alice_20)
watch_bob_20 = weakref.ref(bob_20)
stop_alice_20 = registry_20.subscribe(alice_20)
stop_bob_20 = registry_20.subscribe(bob_20)
assert registry_20.count == 2
registry_20.publish('ping')
assert delivery_log_20 == ['Alice:ping']
assert len(registry_20.errors) == 1
assert registry_20.errors[0][1] == 'ValueError'
print('Delivered to Alice; Bob error isolated.')

Delivered to Alice; Bob error isolated.


### Step 3 — Exercise explicit unsubscribe and collection

Unsubscription is idempotent. A saved token must not keep the client alive. A client with only registry observers is automatically unregistered at death.

In [54]:
stop_alice_20()
stop_alice_20()
assert registry_20.count == 1
del alice_20
assert watch_alice_20() is None
assert registry_20.count == 1

del bob_20
assert watch_bob_20() is None
assert registry_20.count == 0
stop_bob_20()  # no-op after expiry
print('After unsubscribe and expiry:', registry_20.count)

After unsubscribe and expiry: 0


### Step 4 — Confirm publication remains safe with zero clients

This last check tests the public API rather than relying solely on internal state.

In [55]:
registry_20.publish('nobody-home')
assert delivery_log_20 == ['Alice:ping']
assert registry_20.count == 0
print('Capstone passed: no retained clients or duplicate deliveries.')

Capstone passed: no retained clients or duplicate deliveries.


### Solution: what the experiment establishes

A non-owning registry needs more than a weak container: it needs explicit ownership rules for callbacks and tokens, a safe single dereference at call time, mutation-safe iteration, well-defined error handling, and tests for expiry and explicit removal. In a concurrently accessed production registry, protect `_entries` with a lock and avoid executing user callbacks while holding it.

---

# Final consolidation — test your mental model

**Question A.** Why can an object still be alive after `del x`? Because other strong edges may exist: defaults, closures, traceback frames, suspended generators, or other owners.

**Question B.** When is a weak reference a strong owner? Never by itself. Its **returned referent** is an ordinary strong reference if you store it or keep it in a local during use.

**Question C.** Can a weak-value cache guarantee its entry will survive until the next request? No. The application must supply strong ownership for residency.

**Question D.** Why use `weakref.finalize` with independent state? So the cleanup registration does not accidentally retain the owner it is supposed to observe.

**Question E.** Can weak collections change while you iterate? Yes. Snapshot iteration can make one traversal stable, at the cost of temporary strong references.

**Question F.** What is the right way to test collection? Observe a weak ref, avoid printing or saving the referent as a notebook result, isolate temporary owners, and explicitly collect unreachable cycles when relevant. Do **not** read freed object memory addresses.

### Further self-directed modifications

Try these without looking up a solution: (1) make the capstone subscriber unsubscribe itself during `publish`; (2) give the capstone registry a client whose `__eq__` returns true for a distinct instance and verify token independence; (3) make the cache factory raise once and verify a second request can retry; (4) add a context-manager protocol to `ResourceLease`; (5) investigate how generator lifetime changes if you `yield` the payload instead of a string.

**Completion criterion:** every assertion from Problem 01 through Problem 20 succeeds when the notebook is run from a fresh kernel, top to bottom.